### All neccesory imports for the milestone 5

In [17]:
from pathlib import Path
import os
import librosa
import torch 
import torch.nn as nn
from torch.utils.data import dataloader, Dataset
import numpy as np
from transformers import  ASTFeatureExtractor, ASTForAudioClassification



In [2]:
Base_dir     = Path("/kaggle/input/competitions/jan-2026-dl-gen-ai-project/messy_mashup")
es50_dir=Base_dir/"ESC-50-master"
MASHUPS_DIR=Base_dir/"meshups"
ESC50_META   = es50_dir / "meta" / "esc50.csv"
test_csv=Base_dir/ "test.csv"
ESC50_AUDIO_DIR=es50_dir/"audio"
STEMS_DIR    = Base_dir / "genres_stems"


GENRES = ["blues","classical","country","disco","hiphop",
          "jazz","metal","pop","reggae","rock"]
STEMS  = ["drums.wav","vocals.wav","bass.wav","other.wav"]

Base_dir.exists()

True

In [3]:
for gener in GENRES:
    genre_path=STEMS_DIR / gener
    print(genre_path)
    if not genre_path.exists():
        print("missing folder")
        continue
    songs=[d for d in genre_path.iterdir() if d.is_dir() ]
    for song in songs:
        stemfiles=list(song.iterdir())
        for stem in stemfiles:
            y,sr= librosa.load(str(stem), sr=None, mono=True)
print(sr)
           # print(y, sr)
            

/kaggle/input/competitions/jan-2026-dl-gen-ai-project/messy_mashup/genres_stems/blues
/kaggle/input/competitions/jan-2026-dl-gen-ai-project/messy_mashup/genres_stems/classical
/kaggle/input/competitions/jan-2026-dl-gen-ai-project/messy_mashup/genres_stems/country
/kaggle/input/competitions/jan-2026-dl-gen-ai-project/messy_mashup/genres_stems/disco
/kaggle/input/competitions/jan-2026-dl-gen-ai-project/messy_mashup/genres_stems/hiphop
/kaggle/input/competitions/jan-2026-dl-gen-ai-project/messy_mashup/genres_stems/jazz
/kaggle/input/competitions/jan-2026-dl-gen-ai-project/messy_mashup/genres_stems/metal
/kaggle/input/competitions/jan-2026-dl-gen-ai-project/messy_mashup/genres_stems/pop
/kaggle/input/competitions/jan-2026-dl-gen-ai-project/messy_mashup/genres_stems/reggae
/kaggle/input/competitions/jan-2026-dl-gen-ai-project/messy_mashup/genres_stems/rock
44100


### Q1): Dataset Splitting Strategy Task:

In [12]:
def build_song_pool():
    all_recipes = []
    for genre in GENRES:
        genre_path = STEMS_DIR / genre
        songs = sorted([d for d in genre_path.iterdir() if d.is_dir()])
        print(f"   {genre:12s} → {len(songs)} songs")
        for song_dir in songs:
            all_recipes.append({             # one dict per song
                "path":  song_dir,
                "genre": genre,
                "label_id": GENRES.index(genre)
            })
    return all_recipes
 

recipes=build_song_pool()
train_recipes, val_recipes = train_test_split(
    recipes, test_size=0.2, shuffle=True, random_state=42
)
print(len(val_recipes))
train_recipes[0]

   blues        → 100 songs
   classical    → 100 songs
   country      → 100 songs
   disco        → 100 songs
   hiphop       → 100 songs
   jazz         → 100 songs
   metal        → 100 songs
   pop          → 100 songs
   reggae       → 100 songs
   rock         → 100 songs
200


{'path': PosixPath('/kaggle/input/competitions/jan-2026-dl-gen-ai-project/messy_mashup/genres_stems/blues/blues.00029'),
 'genre': 'blues',
 'label_id': 0}

### Q2) On-the-Fly Mixing Dimensions Task:

In [13]:
noise_files = sorted(ESC50_AUDIO_DIR.glob("*.wav"))
print(f"Noise files available: {len(noise_files)}")   # ✅ 2000

Noise files available: 2000


In [14]:
def load_and_pad(path):
    audio, _ = librosa.load(path, sr=16000, duration=10, mono=True)
    if len(audio) < TARGET_LEN:
        audio = np.pad(audio, (0, TARGET_LEN - len(audio)))
    else:
        audio = audio[:TARGET_LEN]
    return audio.astype(np.float32)    #

In [15]:
def make_mix(recipe, noise_weight=0.2):
    stem_dir = recipe["path"]          # genres_stems/blues/blues.00000/

    # Load exactly the 4 stems in defined order
    stems = [load_and_pad(stem_dir / stem) for stem in STEMS]
    mix   = np.sum(stems, axis=0)      # (160000,)

    # Add random ESC-50 noise
    noise = load_and_pad(random.choice(noise_files))
    mix   = mix + noise_weight * noise

    # Peak normalize
    mix = mix / (np.max(np.abs(mix)) + 1e-9)
    return mix.astype(np.float32)    

### Question 3: Hugging Face Feature Extractor Shape Task: 

In [21]:
ast="MIT/ast-finetuned-audioset-10-10-0.4593"
ast_feature_extractor=ASTFeatureExtractor.from_pretrained(ast)
mix = np.ones(160000)
inputs = ast_feature_extractor(mix, sampling_rate=16000, return_tensors="pt")
tensor = inputs["input_values"].squeeze(0)

print(tensor.shape)  


torch.Size([1024, 128])


### Q4: Model Architecture Initialization Task: 

In [24]:
ID2LABEL={i:g for i,g in enumerate(GENRES)}
LABEL2ID={g:i for i,g in enumerate(GENRES)}
model = ASTForAudioClassification.from_pretrained(
    "MIT/ast-finetuned-audioset-10-10-0.4593",
    num_labels=10,
    label2id=LABEL2ID,
    id2label=ID2LABEL,
    ignore_mismatched_sizes=True   # ← replaces the 527-class head with 10-class head
)


trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
total_params     = sum(p.numel() for p in model.parameters())

print(f"Trainable : {trainable_params:,}")
print(f"Total     : {total_params:,}")

Loading weights:   0%|          | 0/203 [00:00<?, ?it/s]

ASTForAudioClassification LOAD REPORT from: MIT/ast-finetuned-audioset-10-10-0.4593
Key                     | Status   |                                                                                        
------------------------+----------+----------------------------------------------------------------------------------------
classifier.dense.bias   | MISMATCH | Reinit due to size mismatch ckpt: torch.Size([527]) vs model:torch.Size([10])          
classifier.dense.weight | MISMATCH | Reinit due to size mismatch ckpt: torch.Size([527, 768]) vs model:torch.Size([10, 768])

Notes:
- MISMATCH	:ckpt weights were loaded, but they did not match the original empty weight shapes.


Trainable : 86,196,490
Total     : 86,196,490


### Q5): Inference Normalization Math Task:

In [27]:
y_test = np.array([-0.85, 0.40, 0.20, -0.10])

y = y_test / (np.max(np.abs(y_test)) + 1e-9)  
y

array([-1.        ,  0.47058823,  0.23529412, -0.11764706])